# Final Project: Data Science in Cyber
**Topic**: Phishing Detection
**Source**: [Phishing Website Detection Repository](https://github.com/sujeetgund/phishing-website-detection)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             matthews_corrcoef, roc_auc_score, confusion_matrix, classification_report)

# Setting random seeds for reproducibility
np.random.seed(42)
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
- **Data loading**
- **Data inspection**
- **Data size, feature types**
- **Missing value analysis**

In [ ]:
# Load the dataset
# Assuming the data is in 'data/phishingData.csv'
data_path = 'data/phishingData.csv'
df = pd.read_csv(data_path)

# Standardize column names to lower case
df.columns = df.columns.str.lower()
print(f"Data shape: {df.shape}")

In [ ]:
# Data inspection and feature types
display(df.head())
display(df.info())

In [ ]:
# Missing value analysis
missing_values = df.isnull().sum()
print("Missing values in each column:\n", missing_values[missing_values > 0])
print(f"Total missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for single value or duplicated features
single_value_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Columns with a single value: {single_value_cols}")

duplicated_rows = df.duplicated().sum()
print(f"Duplicated rows: {duplicated_rows}")
# Note: In phishing datasets, duplicated rows might exist due to the same features extracted from similar URLs. 
# We'll drop duplicates to prevent data leakage between train/test.
df = df.drop_duplicates()
print(f"Data shape after dropping duplicates: {df.shape}")

## 2. Exploratory Data Analysis (EDA)
- **Feature distributions**
- **Crosstab/Group-by Analysis**
- **Correlation Analysis**
- **Class Imbalance**

In [ ]:
# Class imbalance analysis
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='result')
plt.title('Distribution of Target Variable (Result)')
plt.show()

# 1 indicates phishing, -1 indicates legitimate
print(df['result'].value_counts(normalize=True))

In [ ]:
# Correlation Analysis using Spearman (robust to non-normal, ordinal data like ours which are -1, 0, 1)
plt.figure(figsize=(15, 12))
corr = df.corr(method='spearman')
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title('Spearman Correlation Heatmap')
plt.show()

In [ ]:
# Identify features most correlated with the target
target_corr = corr['result'].sort_values(ascending=False)
print("Top positive correlations with target:\n", target_corr.head(10))
print("\nTop negative correlations with target:\n", target_corr.tail(10))

In [ ]:
# Crosstab analysis example: URL_Length vs Result
ct = pd.crosstab(df['url_length'], df['result'], normalize='index')
ct.plot(kind='bar', stacked=True)
plt.title('URL Length vs Phishing Result')
plt.ylabel('Proportion')
plt.show()

## 3. Feature Engineering
- **Encoding categorical variables**
- **Feature selection**
- **Feature scaling**

In [ ]:
# The features are mostly encoded as -1, 0, 1 which are categorical but represented ordinally.
# We will define X and y.
X = df.drop(columns=['result'])
y = df['result']

# Encode the target: map -1 to 0 and 1 to 1 for standard binary classification metrics
y = y.map({-1: 0, 1: 1})

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

In [ ]:
# Feature scaling is generally less impactful for tree-based models, but crucial for Logistic Regression/SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Model Training
We will train at least two models: Random Forest and Logistic Regression.

In [ ]:
# 1. Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# 2. Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

## 5. Evaluation & Error Analysis
- **Evaluation Metrics**
- **Error Analysis**

In [ ]:
def evaluate_model(y_true, y_pred, y_probs, model_name):
    print(f"--- Evaluation for {model_name} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_true, y_pred):.4f}")
    print(f"MCC      : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_probs):.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

evaluate_model(y_test, rf_preds, rf_probs, "Random Forest")
evaluate_model(y_test, lr_preds, lr_probs, "Logistic Regression")

In [ ]:
# Error Analysis for Random Forest
# Trade-off between False Positives and False Negatives
# False Positive: Legitimate website flagged as Phishing (User annoyance, blocked access)
# False Negative: Phishing website flagged as Legitimate (Severe security risk, data loss)
# In cybersecurity, we often prefer minimizing False Negatives.

# Let's look at False Negatives
errors_df = X_test.copy()
errors_df['Actual'] = y_test
errors_df['Predicted_RF'] = rf_preds

fn_df = errors_df[(errors_df['Actual'] == 1) & (errors_df['Predicted_RF'] == 0)]
fp_df = errors_df[(errors_df['Actual'] == 0) & (errors_df['Predicted_RF'] == 1)]

print(f"Total False Negatives (Phishing missed): {len(fn_df)}")
print(f"Total False Positives (Legit blocked): {len(fp_df)}")

# Analyzing pattern in False Negatives
display(fn_df.describe())